# TP 2 

In [1]:
corpus = [
    "I loved the camera! It takes amazing photos even in low light.",
    "Totally disappointed. Battery life is terrible after a week of use.",
    "Great value for money. Highly recommended for casual users!",
    "Worst product ever. I returned it after two days.",
    "Fast delivery, good packaging, but the screen is too small."
]


### IMPORT


In [4]:
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# Téléchargements nécessaires
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# SpaCy
spacy_model = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Yann/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Yann/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Nettoyage


In [2]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # liens
    text = re.sub(r"@\w+|#\w+", '', text)  # mentions et hashtags
    text = text.translate(str.maketrans('', '', string.punctuation))  # ponctuation
    return text

cleaned_corpus = [clean_text(sentence) for sentence in corpus]
print(cleaned_corpus)


['i loved the camera it takes amazing photos even in low light', 'totally disappointed battery life is terrible after a week of use', 'great value for money highly recommended for casual users', 'worst product ever i returned it after two days', 'fast delivery good packaging but the screen is too small']


### Supressions stop words

In [5]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]

no_stopwords_corpus = [remove_stopwords(text) for text in cleaned_corpus]
print(no_stopwords_corpus)


[['loved', 'camera', 'takes', 'amazing', 'photos', 'even', 'low', 'light'], ['totally', 'disappointed', 'battery', 'life', 'terrible', 'week', 'use'], ['great', 'value', 'money', 'highly', 'recommended', 'casual', 'users'], ['worst', 'product', 'ever', 'returned', 'two', 'days'], ['fast', 'delivery', 'good', 'packaging', 'screen', 'small']]


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


###  Tokenisation (nltk vs spaCy)

nltk

In [6]:
print("Tokenisation nltk :", word_tokenize(cleaned_corpus[0]))


Tokenisation nltk : ['i', 'loved', 'the', 'camera', 'it', 'takes', 'amazing', 'photos', 'even', 'in', 'low', 'light']


spacy


In [7]:
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp(cleaned_corpus[0])
print("Tokenisation spacy :", [token.text for token in doc])


Tokenisation spacy : ['i', 'loved', 'the', 'camera', 'it', 'takes', 'amazing', 'photos', 'even', 'in', 'low', 'light']


###  Stemming (PorterStemmer)

In [8]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

example = no_stopwords_corpus[0]
stemmed = [stemmer.stem(word) for word in example]
print("Stemming :", stemmed)


Stemming : ['love', 'camera', 'take', 'amaz', 'photo', 'even', 'low', 'light']


###   Lemmatization (spaCy)

In [9]:
doc = nlp(' '.join(example))
lemmatized = [token.lemma_ for token in doc]
print("Lemmatization :", lemmatized)


Lemmatization : ['love', 'camera', 'take', 'amazing', 'photo', 'even', 'low', 'light']


✅ Avantages du stemming :
✅ Rapide à exécuter (règles simples, pas de modèle linguistique).

✅ Réduction brutale efficace pour certains modèles (réduction du vocabulaire).

❌ Inconvénients :
❌ Peut produire des racines incorrectes ou illisibles ("better" → "better", "running" → "run", "flying" → "fli").

❌ Pas sensible au contexte → peut mélanger des mots différents.

En résumé :
Stemming = coupe brutalement → efficace mais parfois "bête".

Lemmatization = base grammaticale correcte → plus fiable mais plus lente.



 ### Vocabulaire

In [10]:
# Vocabulaire brut
raw_vocab = set(word.lower() for sentence in corpus for word in word_tokenize(sentence))
print("Taille vocabulaire brut :", len(raw_vocab))

# Vocabulaire nettoyé + lemmatisé
lemm_vocab = set()
for sent in cleaned_corpus:
    doc = nlp(sent)
    lemm_vocab.update([token.lemma_ for token in doc if token.lemma_ not in stop_words and token.lemma_ not in string.punctuation])

print("Taille vocabulaire lemmatisé :", len(lemm_vocab))


Taille vocabulaire brut : 48
Taille vocabulaire lemmatisé : 35


### Fréquence des mots

In [11]:
from collections import Counter

all_tokens = []
for text in cleaned_corpus:
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    all_tokens.extend(tokens)

freq = Counter(all_tokens)
print("Top 5 mots fréquents :", freq.most_common(5))


Top 5 mots fréquents : [('loved', 1), ('camera', 1), ('takes', 1), ('amazing', 1), ('photos', 1)]


### Importance du contexte (lemmatisation)

In [12]:
text = "He saw a bat flying in the dark."
doc = nlp(text)
print([(token.text, token.lemma_) for token in doc])


[('He', 'he'), ('saw', 'see'), ('a', 'a'), ('bat', 'bat'), ('flying', 'fly'), ('in', 'in'), ('the', 'the'), ('dark', 'dark'), ('.', '.')]


Application pratique (classification sentiment)
Quels prétraitements garder pour analyser des sentiments (positif/négatif) ?

Recommandés :
✅ Mise en minuscule

✅ Suppression ponctuation

✅ Suppression stopwords

✅ Lemmatization (meilleur que stemming)

✅ Optionnel : conserver les n-grams ("not good", "very bad")

Non recommandés :
❌ Stemming (perd des nuances utiles au sens)

❌ Retrait de mots comme "not" (essentiels au sens du sentiment !)